In [1]:
import pandas as pd
train_df = pd.read_csv(r"C:\Users\chill\OneDrive\Documents\ML-DS Competition\StudentHealthRisk\datasets\train.csv")
test_df = pd.read_csv(r"C:\Users\chill\OneDrive\Documents\ML-DS Competition\StudentHealthRisk\datasets\test.csv")

In [2]:
train_df.shape

(690088, 15)

In [3]:
test_df.shape

(295753, 14)

In [3]:
train_df = train_df.iloc[:,1:]
train_df.columns

Index(['health_condition', 'sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake', 'diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender'],
      dtype='object')

In [4]:
y = train_df['health_condition']
X = train_df.drop("health_condition", axis=1)
X.columns

Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake', 'diet_type',
       'stress_level', 'sleep_quality', 'physical_activity_level',
       'smoking_alcohol', 'gender'],
      dtype='object')

In [5]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()
print("numerical cols: \n", numerical_features)
print("\ncategorical cols: \n", categorical_features)

numerical cols: 
 ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

categorical cols: 
 ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [7]:
print("before filling null values for X: ", X.isnull().sum().sum())
print("before filling null values for test: ", test_df.isnull().sum().sum())
for col in categorical_features:

    X[col] = X[col].fillna("Missing")
    test_df[col] = test_df[col].fillna("Missing")

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())
    test_df[col] = test_df[col].fillna(test_df[col].median())

print("after filling null values for X: ", X.isnull().sum().sum())
print("after filling null values for X: ", test_df.isnull().sum().sum())

before filling null values for X:  442595
before filling null values for test:  189684
after filling null values for X:  0
after filling null values for X:  0


In [8]:
for cols in categorical_features:
    print(f'{cols}" {X[cols].unique()}')

diet_type" ['veg' 'non-veg' 'balanced' 'Missing']
stress_level" ['high' 'low' 'Missing' 'medium']
sleep_quality" ['average' 'poor' 'Missing' 'good']
physical_activity_level" ['sedentary' 'moderate' 'active' 'Missing']
smoking_alcohol" ['yes' 'occasional' 'Missing' 'no']
gender" ['female' 'other' 'male' 'Missing']


In [9]:
X.isnull().sum()

sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

In [10]:
def feature_engineering(df):
    import numpy as np
    df = df.copy()

    # ------------------------
    # Ratios
    # ------------------------

    if {"step_count", "exercise_duration"} <= set(df.columns):
        df["steps_per_min"] = df["step_count"] / (df["exercise_duration"] + 1)

    if {"calorie_expenditure", "step_count"} <= set(df.columns):
        df["calorie_per_step"] = df["calorie_expenditure"] / (df["step_count"] + 1)

    if {"water_intake", "bmi"} <= set(df.columns):
        df["water_bmi"] = df["water_intake"] /(df["bmi"] + 1)

    # ------------------------
    # Multiplication Features
    # ------------------------

    if {"exercise_duration", "calorie_expenditure"} <= set(df.columns):
        df["exercise_calorie"] = (
            df["exercise_duration"] *
            df["calorie_expenditure"]
        )

    if {"heart_rate", "bmi"} <= set(df.columns):
        df["heart_bmi"] = (
            df["heart_rate"] *
            df["bmi"]
        )

    
    # ------------------------
    # Non-linear Features
    # ------------------------

    if "step_count" in df.columns:
        df["log_steps"] = np.log1p(df["step_count"])

    if "calorie_expenditure" in df.columns:
        df["sqrt_calories"] = np.sqrt(df["calorie_expenditure"])

    if "bmi" in df.columns:
        df["bmi_squared"] = df["bmi"] ** 2

    return df

In [11]:
X = feature_engineering(X)
test = feature_engineering(test_df)

In [12]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   sleep_duration           690088 non-null  float64
 1   heart_rate               690088 non-null  float64
 2   bmi                      690088 non-null  float64
 3   calorie_expenditure      690088 non-null  float64
 4   step_count               690088 non-null  float64
 5   exercise_duration        690088 non-null  float64
 6   water_intake             690088 non-null  float64
 7   diet_type                690088 non-null  object 
 8   stress_level             690088 non-null  object 
 9   sleep_quality            690088 non-null  object 
 10  physical_activity_level  690088 non-null  object 
 11  smoking_alcohol          690088 non-null  object 
 12  gender                   690088 non-null  object 
 13  steps_per_min            690088 non-null  float64
 14  calo

In [13]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

cat_features_index = [
    X.columns.get_loc(col)
    for col in categorical_features
]

In [15]:
cat_features_index

[7, 8, 9, 10, 11, 12]

In [14]:
categorical_features

['diet_type',
 'stress_level',
 'sleep_quality',
 'physical_activity_level',
 'smoking_alcohol',
 'gender']

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import numpy as np
from catboost import CatBoostClassifier

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)
oof = np.zeros(len(X))
test_pred = []
params = {
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",
    "iterations": 1000,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 5,
    "subsample": 0.8,
    "random_strength": 2,
    "min_data_in_leaf": 30,
    "bootstrap_type": "Bernoulli",
    "random_seed": 42,
    "verbose": 200,
    "early_stopping_rounds": 300
}


In [19]:
scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f"\nFold {fold+1}")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(**params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features_index,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    pred = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid,pred)
    print(score)
    scores.append(score)
    oof[valid_idx] = pred
    test_pred.append(
        model.predict_proba(test)
    )


Fold 1
0:	learn: 1.0481894	test: 1.0479548	best: 1.0479548 (0)	total: 524ms	remaining: 8m 43s
200:	learn: 0.0973442	test: 0.0955703	best: 0.0955703 (200)	total: 1m 45s	remaining: 6m 59s
400:	learn: 0.0924446	test: 0.0914875	best: 0.0914871 (398)	total: 3m 27s	remaining: 5m 9s
600:	learn: 0.0907713	test: 0.0904959	best: 0.0904959 (600)	total: 5m 13s	remaining: 3m 27s
800:	learn: 0.0897103	test: 0.0900563	best: 0.0900563 (800)	total: 7m 6s	remaining: 1m 45s
999:	learn: 0.0886383	test: 0.0897459	best: 0.0897459 (999)	total: 8m 59s	remaining: 0us

bestTest = 0.08974591052
bestIteration = 999

0.8685663228469256


ValueError: shape mismatch: value array of shape (230030,1) could not be broadcast to indexing result of shape (230030,)

In [ ]:

y = y.map({
    "fit":0,
    "unhealthy":1,
    "at-risk":2
})
y.head(10)

0    1
1    2
2    1
3    1
4    2
5    2
6    2
7    2
8    1
9    2
Name: health_condition, dtype: int64

In [ ]:
print(X.head(3))

In [ ]:
import catboost

In [ ]:
# feature engineering : 

X["activity_ratio"] = X["step_count"] / X["exercise_duration"]

X["calorie_per_step"] = X["calorie_expenditure"] / X["step_count"]

X["water_per_bmi"] = X["water_intake"] / X["bmi"]

X["sleep_efficiency"] = X["sleep_duration"] * X["sleep_quality"]

activity_score = exercise_duration * physical_activity_level

stress_sleep = stress_level * sleep_duration

exercise_calorie =
exercise_duration * calorie_expenditure